# HKG CAN-FD BSM Regression Test

This notebook validates that changing the BSM fingerprint check from `0x1E5` (BLINDSPOTS_FRONT_CORNER_1) to `0x1BA` (BLINDSPOTS_REAR_CORNERS) doesn't break BSM detection for any existing CAN-FD cars.

**Background:** The original code checked for `0x1E5` to enable BSM, but the actual BSM indicator signals (FL_INDICATOR, FR_INDICATOR) are in message `0x1BA`. This test verifies all cars with `0x1E5` also have `0x1BA`.

In [1]:
import sys
import os
import requests
import bz2

# Add openpilot root to path for cereal imports
OPENPILOT_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), "../../.."))
if OPENPILOT_ROOT not in sys.path:
    sys.path.insert(0, OPENPILOT_ROOT)

from opendbc.car.hyundai.values import CAR, HyundaiFlags
from opendbc.car.tests.routes import routes as test_routes
from cereal import log as capnp_log

# BSM CAN addresses
OLD_BSM_ADDR = 0x1e5  # BLINDSPOTS_FRONT_CORNER_1 (old check)
NEW_BSM_ADDR = 0x1ba  # BLINDSPOTS_REAR_CORNERS (new check)

kj/filesystem-disk-unix.c++:1734: warning: PWD environment variable doesn't match current directory; pwd = /Users/zach/projects/openpilot


In [2]:
def download_route(route_str, segment=0):
    """Download route data from public CI (old format routes only)"""
    if "|" in route_str:
        dongle, date_time = route_str.split("|")
        url = f"https://commadataci.blob.core.windows.net/openpilotci/{dongle}/{date_time}/{segment}/rlog.bz2"
        r = requests.get(url, timeout=60)
        if r.status_code == 200:
            return bz2.decompress(r.content)
    return None

# Get all CAN-FD test routes (public CI only)
canfd_routes = []
for route in test_routes:
    if hasattr(route.car_model, 'config') and route.car_model.config.flags & HyundaiFlags.CANFD:
        if "|" in route.route:  # Only old format (public CI) routes
            canfd_routes.append(route)

print(f"Found {len(canfd_routes)} CAN-FD test routes to analyze")

Found 24 CAN-FD test routes to analyze


In [3]:
print(f"=== BSM Regression Test ===\n")
print(f"OLD check: 0x{OLD_BSM_ADDR:03X} (BLINDSPOTS_FRONT_CORNER_1)")
print(f"NEW check: 0x{NEW_BSM_ADDR:03X} (BLINDSPOTS_REAR_CORNERS)\n")

results = []
tested_platforms = set()

for route in canfd_routes:
    platform = route.car_model.name
    route_str = route.route
    segment = route.segment if hasattr(route, 'segment') and route.segment else 0
    
    # Skip duplicate platforms
    if platform in tested_platforms:
        continue
    
    print(f"{platform}...", end=" ", flush=True)
    
    try:
        data = download_route(route_str, segment)
        if data is None:
            print("SKIP (can't download)")
            continue
        
        events = list(capnp_log.Event.read_multiple_bytes(data))
        
        # Get carParams
        CP = None
        for event in events:
            if event.which() == 'carParams':
                CP = event.carParams
                break
        
        if CP is None:
            print("SKIP (no carParams)")
            continue
        
        # Check ALL buses for BSM addresses
        has_old = False
        has_new = False
        for event in events[:5000]:
            if event.which() == 'can':
                for m in event.can:
                    if m.address == OLD_BSM_ADDR:
                        has_old = True
                    if m.address == NEW_BSM_ADDR:
                        has_new = True
                if has_old and has_new:
                    break
        
        tested_platforms.add(platform)
        
        # Determine status
        if has_old and has_new:
            status = "BOTH present ✓"
        elif has_new and not has_old:
            status = "Only 0x1BA (improved)"
        elif has_old and not has_new:
            status = "⚠️ Only 0x1E5 (REGRESSION!)"
        else:
            status = "Neither (no BSM HW?)"
        
        results.append({
            'platform': platform,
            'has_0x1e5': has_old,
            'has_0x1ba': has_new,
            'status': status,
            'current_bsm': CP.enableBsm,
        })
        
        print(f"0x1E5={has_old!s:<5} 0x1BA={has_new!s:<5} → {status}")
        
    except Exception as e:
        print(f"ERROR: {e}")

=== BSM Regression Test ===

OLD check: 0x1E5 (BLINDSPOTS_FRONT_CORNER_1)
NEW check: 0x1BA (BLINDSPOTS_REAR_CORNERS)

GENESIS_GV60_EV_1ST_GEN... 0x1E5=True  0x1BA=True  → BOTH present ✓
GENESIS_GV70_1ST_GEN... 0x1E5=True  0x1BA=True  → BOTH present ✓
HYUNDAI_SANTA_CRUZ_1ST_GEN... 0x1E5=True  0x1BA=True  → BOTH present ✓
KIA_CARNIVAL_4TH_GEN... 0x1E5=True  0x1BA=True  → BOTH present ✓
HYUNDAI_STARIA_4TH_GEN... 0x1E5=True  0x1BA=True  → BOTH present ✓
HYUNDAI_TUCSON_4TH_GEN... 0x1E5=True  0x1BA=True  → BOTH present ✓
KIA_SORENTO_4TH_GEN... 0x1E5=True  0x1BA=True  → BOTH present ✓
KIA_SORENTO_HEV_4TH_GEN... 0x1E5=True  0x1BA=True  → BOTH present ✓
HYUNDAI_IONIQ_5... 0x1E5=True  0x1BA=True  → BOTH present ✓
HYUNDAI_IONIQ_6... 0x1E5=True  0x1BA=True  → BOTH present ✓
HYUNDAI_KONA_EV_2ND_GEN... 0x1E5=True  0x1BA=True  → BOTH present ✓
KIA_EV6... 0x1E5=True  0x1BA=True  → BOTH present ✓
KIA_K8_HEV_1ST_GEN... 0x1E5=True  0x1BA=True  → BOTH present ✓
KIA_NIRO_EV_2ND_GEN... 0x1E5=False 0x1BA=Fal

In [4]:
print(f"\n{'='*70}")
print(f"=== SUMMARY ===\n")

both = [r for r in results if "BOTH" in r['status']]
improved = [r for r in results if "improved" in r['status']]
regression = [r for r in results if "REGRESSION" in r['status']]
neither = [r for r in results if "Neither" in r['status']]

print(f"Platforms tested: {len(results)}")
print(f"  Both 0x1E5 & 0x1BA:    {len(both)}")
print(f"  Only 0x1BA:            {len(improved)}")
print(f"  Only 0x1E5:            {len(regression)}")
print(f"  Neither:               {len(neither)}")

if regression:
    print(f"\n⚠️  REGRESSIONS FOUND - these cars would lose BSM!")
    for r in regression:
        print(f"  {r['platform']}")
else:
    print(f"\n✓ NO REGRESSIONS!")
    print(f"  All cars with 0x1E5 also have 0x1BA")
    print(f"  Safe to change BSM check from 0x1E5 to 0x1BA")


=== SUMMARY ===

Platforms tested: 17
  Both 0x1E5 & 0x1BA:    16
  Only 0x1BA:            0
  Only 0x1E5:            0
  Neither:               1

✓ NO REGRESSIONS!
  All cars with 0x1E5 also have 0x1BA
  Safe to change BSM check from 0x1E5 to 0x1BA


In [5]:
# Validate that 0x1BA actually contains BSM activity (non-zero indicators)
# This proves the signals are being broadcast, not just empty messages

from opendbc.can.parser import CANParser

print("=== BSM Signal Activity Check ===\n")
print("Checking if FL_INDICATOR / FR_INDICATOR in 0x1BA show actual BSM activity...\n")

bsm_activity_results = []

for route in canfd_routes:
    platform = route.car_model.name
    route_str = route.route
    segment = route.segment if hasattr(route, 'segment') and route.segment else 0
    
    if platform not in tested_platforms:
        continue
    
    # Check if this platform has 0x1BA
    result = next((r for r in results if r['platform'] == platform), None)
    if not result or not result['has_0x1ba']:
        continue
    
    print(f"{platform}...", end=" ", flush=True)
    
    try:
        data = download_route(route_str, segment)
        if data is None:
            print("SKIP")
            continue
        
        events = list(capnp_log.Event.read_multiple_bytes(data))
        
        # Decode FL_INDICATOR and FR_INDICATOR from 0x1BA
        # DBC: SG_ FL_INDICATOR : 46|6@0+ and SG_ FR_INDICATOR : 54|6@0+
        # These are big-endian 6-bit signals
        
        fl_max = 0
        fr_max = 0
        msg_count = 0
        
        for event in events:
            if event.which() == 'can':
                for m in event.can:
                    if m.address == NEW_BSM_ADDR and len(m.dat) >= 8:
                        msg_count += 1
                        dat = bytes(m.dat)
                        
                        # Extract FL_INDICATOR (start bit 46, 6 bits, big-endian)
                        # Byte 5 bits 6-1 (46 = byte 5, bit 6 as MSB)
                        fl = (dat[5] >> 1) & 0x3F
                        
                        # Extract FR_INDICATOR (start bit 54, 6 bits, big-endian) 
                        # Byte 6 bits 6-1
                        fr = (dat[6] >> 1) & 0x3F
                        
                        fl_max = max(fl_max, fl)
                        fr_max = max(fr_max, fr)
        
        has_activity = fl_max > 0 or fr_max > 0
        status = f"FL={fl_max}, FR={fr_max}" if has_activity else "No activity in route"
        
        bsm_activity_results.append({
            'platform': platform,
            'fl_max': fl_max,
            'fr_max': fr_max,
            'has_activity': has_activity,
            'msg_count': msg_count,
        })
        
        indicator = "✓ ACTIVE" if has_activity else "(no BSM trigger in route)"
        print(f"{msg_count} msgs, FL_max={fl_max}, FR_max={fr_max} {indicator}")
        
    except Exception as e:
        print(f"ERROR: {e}")

print(f"\n--- Activity Summary ---")
active = [r for r in bsm_activity_results if r['has_activity']]
print(f"Platforms with BSM activity in route: {len(active)}/{len(bsm_activity_results)}")
if active:
    print("These cars show non-zero BSM indicators in 0x1BA:")
    for r in active:
        print(f"  {r['platform']}: FL={r['fl_max']}, FR={r['fr_max']}")
print("\nNote: No activity just means BSM wasn't triggered during the test route.")

=== BSM Signal Activity Check ===

Checking if FL_INDICATOR / FR_INDICATOR in 0x1BA show actual BSM activity...

GENESIS_GV60_EV_1ST_GEN... 2400 msgs, FL_max=0, FR_max=0 (no BSM trigger in route)
GENESIS_GV70_1ST_GEN... 2450 msgs, FL_max=0, FR_max=0 (no BSM trigger in route)
HYUNDAI_SANTA_CRUZ_1ST_GEN... 2452 msgs, FL_max=0, FR_max=0 (no BSM trigger in route)
KIA_CARNIVAL_4TH_GEN... 2476 msgs, FL_max=0, FR_max=0 (no BSM trigger in route)
KIA_CARNIVAL_4TH_GEN... 2454 msgs, FL_max=0, FR_max=0 (no BSM trigger in route)
HYUNDAI_STARIA_4TH_GEN... 2468 msgs, FL_max=0, FR_max=0 (no BSM trigger in route)
HYUNDAI_TUCSON_4TH_GEN... 2464 msgs, FL_max=0, FR_max=0 (no BSM trigger in route)
HYUNDAI_TUCSON_4TH_GEN... 2450 msgs, FL_max=0, FR_max=0 (no BSM trigger in route)
KIA_SORENTO_4TH_GEN... 2400 msgs, FL_max=0, FR_max=0 (no BSM trigger in route)
KIA_SORENTO_HEV_4TH_GEN... 2400 msgs, FL_max=0, FR_max=0 (no BSM trigger in route)
KIA_SORENTO_HEV_4TH_GEN... 2448 msgs, FL_max=0, FR_max=0 (no BSM trigg